# Fabric Performance Benchmarking

This notebook analyses the results of benchmarking different data processing engines on Microsoft Fabric. We compare Pandas, PySpark, Polars, and DuckDB across various compute configurations to understand trade-offs between execution time, cost, and resource utilisation.

## Setup and Data Loading

In [2]:
import polars as pl
from azure.identity import InteractiveBrowserCredential
import plotly.express as px

In [3]:
pl.Config.set_tbl_rows(20)

polars.config.Config

In [4]:
# Set up a colour pallete for consistency across charts using endjin colour way
my_palette = ["#84BD00", "#E87722", "#41B6E6", "#31114A", "#A72B2A", "#000000"]
px.defaults.color_discrete_sequence = my_palette

# Default template (controls overall styling)
px.defaults.template = "plotly"

# Default dimensions
px.defaults.height = 800

In [5]:
credential = InteractiveBrowserCredential()

In [6]:
token = credential.get_token("https://storage.azure.com/.default")

In [7]:
# Pre-requisities are to create a Fabric Workspace with a lakehouse, putting names here:
WORKSPACE_NAME = "fabric_performance_benchmark_workspace"
LAKEHOUSE_NAME = "fabric_performance_benchmark_lakehouse"

In [8]:
# Helper function to create base ABFSS path based on workspace and lakehouse name
def construct_base_abfss_path(workspace_name: str, lakehouse_name: str) -> str:
    """Construct the base ABFSS path for a given workspace and lakehouse."""
    # Because it is a URL, replace spaces with %20
    workspace_name = workspace_name.replace(" ", "%20")
    lakehouse_name = lakehouse_name.replace(" ", "%20")
    return f"abfss://{workspace_name}@onelake.dfs.fabric.microsoft.com/{lakehouse_name}.Lakehouse"

# Helper function to create storage options that enable data tools to authenticate and interact with onelake storage
def create_storage_options() -> dict:
    return {
        "bearer_token": token.token,
        "use_fabric_endpoint": "true"
    }

In [9]:
benchmarks_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/benchmarks"

stages_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/stages"

configurations_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/configurations"

benchmark_analytics_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/benchmark_analytics"

In [10]:
storage_options = create_storage_options()

In [11]:
benchmarks = pl.read_delta(benchmark_analytics_path, storage_options=storage_options)

In [12]:
sorted(benchmarks["configuration"].unique().to_list())

['01 executors 04/04 cores 28g/28g memory',
 '01 executors 08/08 cores 56g/56g memory',
 '02 executors 04/04 cores 28g/28g memory',
 '02 executors 08/08 cores 56g/56g memory',
 '02 vCores',
 '04 executors 04/04 cores 28g/28g memory',
 '04 executors 08/08 cores 56g/56g memory',
 '04 vCores',
 '08 vCores',
 '16 vCores',
 '32 vCores']

In [13]:
# Filter to only the configuration we care about for this analysis, as there are some other runs in the data that are not relevant
benchmarks = (
    benchmarks
    .filter(pl.col("configuration").is_in(
            [
                # 1 CU
                '02 vCores',
                # 2 CU
                '04 vCores',
                # 4 CU
                '08 vCores',
                '01 executors 04/04 cores 28g/28g memory',
                # 6 CU
                # '02 executors 04/04 cores 28g/28g memory',
                # 8 CU
                '16 vCores',
                '01 executors 08/08 cores 56g/56g memory',
                # 10 CU
                # '04 executors 04/04 cores 28g/28g memory',
                # 12 CU
                '02 executors 08/08 cores 56g/56g memory',
                # 16 CU
                '32 vCores',
                # 20 CU
                '04 executors 08/08 cores 56g/56g memory',
            ]
        )
    )
)

In [14]:
benchmarks.sample(5)

order,platform,configuration,workload_name,run_timestamp,stage_name,stage_time,cpu_count,cpu_usage,memory,memory_usage,stage_time_delta,stage_order,phase,phase_order,configuration_scale,total_v_cores,cu_per_second,cumulative_time
i32,str,str,str,str,str,datetime[μs],i64,f64,f64,f64,f64,i64,str,i64,str,i64,i64,f64
9,"""Fabric Python Notebook""","""16 vCores""","""polars_benchmark""","""20260204_141501""","""read_dates""",2026-02-04 14:18:09.384104,16,0.9,125.54343,9.0,1.14,9,"""read_and_summarise""",4,"""40""",16,8,188.38
3,"""Fabric Python Notebook""","""32 vCores""","""duckdb_benchmark""","""20260204_121946""","""ingest""",2026-02-04 12:22:14.149526,32,3.5,251.446178,1.6,3.137,3,"""ingest_and_transform""",2,"""50""",32,16,148.148
2,"""Fabric Python Notebook""","""08 vCores""","""polars_benchmark""","""20260203_223134""","""setup""",2026-02-03 22:33:43.215595,8,2.7,62.545181,3.8,129.215,2,"""start_and_setup""",1,"""30""",8,4,129.215
10,"""Fabric PySpark Notebook""","""01 executors 04/04 cores 28g/2…","""pyspark_benchmark""","""20260205_091136""","""join_and_summarise""",2026-02-05 09:16:59.292985,8,89.2,62.545181,42.2,31.734,10,"""read_and_summarise""",4,"""25""",8,4,323.292
2,"""Fabric Python Notebook""","""16 vCores""","""pandas_benchmark""","""20260204_161518""","""setup""",2026-02-04 16:17:26.595018,16,0.9,125.54343,2.4,128.595,2,"""start_and_setup""",1,"""40""",16,8,128.595


## Data Source

The use case is implemented using open data provided by the [UK Land Registry House Price Data open data repository](https://www.gov.uk/government/statistical-data-sets/price-paid-data-downloads).

This data is made available under an [Open Government Licence](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).

The data is provided as a set of CSV files, one per calendar year, which have been downloaded to a Fabric lakehouse.

The data has been collected since 1995, with circa 1 million property sales per year on average—all 30 years of historic data amounts to ~30 million rows and ~5GB of raw CSV data.

This is a typical dataset that we encounter for common enterprise client use cases. The data we are working with will fit in memory for all of the platform configurations we will be testing. We find that many benchmarks focus on processing massive datasets that are rarely encountered in practice.

Our objective is to focus on the constraints we more often encounter:

- **Developer experience** — having processes that run rapidly unlocks significant benefits during the development phase: test suites run quicker, the inner development loop is optimised, and time to value is accelerated. In today's rapidly evolving environment this can yield significant competitive advantages.

- **Total Cost of Ownership (TCO)** — the cost of running data pipelines, both in terms of financial and environmental impact, is becoming a significant factor for many organisations.

## Use Case

The use case mimics a common set of data transformations that you would see on data of this nature. It includes:

1. **Start Up & Set Up** — the overhead of provisioning the platform (Spark or Python) on which the code is going to run, then completing various tasks such as importing Python packages.
2. **Ingestion & Transform** — reading raw data from a set of CSV files, standardising, cleaning, and adding new features.
3. **Write Dimensional Model** — taking different slices of the transformed data and writing them to the lakehouse in Delta format for downstream consumption, in this case as a dimensional model for Power BI.
4. **Read and Summarise** — reading the tables back in and running analysis based on filtering, joining, and summarising the data across different categories.
5. **Benchmark Capture & Clean Up** — at various points in the process above, benchmark timestamps are captured along with other metadata such as memory consumption. These are written to permanent storage in the lakehouse for analysis.

A high level view of this process is captured below:

```mermaid
flowchart TB
A["1. Start Up & Set Up"] --> B["2. Ingestion & Transform"]
B --> C["3. Write Dimensional Model"]
C --> D["4. Read & Summarise"]
D --> E["5. Benchmark Capture & Clean Up"]

```



A more detailed overview of this process is captured in the mermaid diagram below, with the following symbology:
- ⬆️ - reading from lakehouse.
- 🔧 - data wrangling.
- ⬇️ - write to lakehouse.
- 📊 - points in the process where a benchmark timestamp is captured.


```mermaid
flowchart LR

subgraph Phase1["Phase 1 - Start Up & Set Up"]
    direction TB
    A1[Start Up Platform] --> A2[Import Packages]
    A2 --> A3[Set Up Logging]
    A3 --> A4[Define Constants]
    A4 --> A5[Set Up Helper Functions]
    A5 --> A6[Configure Paths]
    A6 --> A7[Initialise BenchmarkManager]
    A7 --> A8["📊 capture: setup"]
end

subgraph Phase2["Phase 2 - Ingest & Transform"]
    direction TB
    B1["⬆️Scan CSV Files"] --> B2["📊 capture: ingest"]
    B2 --> B3["🔧Transform Data"]
    B3 --> B4["Cache Transformed Data"]
    B4 --> B5["📊 capture: transform"]
end

subgraph Phase3["Phase 3 - Write Dimensional Model"]
    direction TB
    C1[🔧Create Prices Table] --> C2[⬇️Write Prices to Delta]
    C2 --> C3["📊 capture: write_prices"]
    
    C3 --> D1["🔧Create Dates Dimension"]
    D1 --> D2["⬇️Write Dates to Delta"]
    D2 --> D3["📊 capture: write_dates"]
    
    D3 --> E1["🔧Create Locations Dimension"]
    E1 --> E2["⬇️Write Locations to Delta"]
    E2 --> E3["📊 capture: write_locations"]
end

subgraph Phase4["Phase 4 - Read & Summarise"]
    direction TB
    F1[⬆️Read Prices from Delta] --> F2["📊 capture: read_prices"]
    F2 --> F3["⬆️Read Dates from Delta"]
    F3 --> F4["📊 capture: read_dates"]
    F4 --> F5["🔧Join Prices ⟕ Dates"]
    F5 --> F6["🔧Aggregate by Month & Property Type"]
    F6 --> F7["🔧Collect & Display Results"]
    F7 --> F8["📊 capture: join_and_summarise"]
end

subgraph Phase5["Phase 5 - Benchmark Capture & Clean Up"]
    direction TB
    G1[⬇️Export Benchmarks] --> G2[Calculate Elapsed Time]
    G2 --> G3[Remove Working Data]
end

Phase1 --> Phase2
Phase2 --> Phase3
Phase3 --> Phase4
Phase4 --> Phase5
```

## Engines

Fabric offers multiple compute engines. For this study we leveraged two:

- **Spark notebooks** — a notebook experience over a Spark cluster hosted on Fabric. Enables polyglot development (Python, R, SQL) over a Spark cluster which is spun up according to your chosen configuration (vCores, memory, number of executor nodes) on demand.

- **Python notebooks** — a relatively new addition to Fabric. Python notebooks provide a single node for execution which can be sized according to a range of pre-defined configurations (vCores and memory). Whilst they are designed for "smaller" workloads, we find that the majority of enterprise use cases can be accommodated on this platform through intelligent choice of tooling and design.

## Workloads

This study compares the same use case implemented across 4 different data processing engines on the Fabric Platform:

1. **Pandas** — the default package for data engineering in Python. The package has a huge following but is only suitable for data volumes which can fit into memory.

2. **PySpark** — the Python API for Apache Spark, a common choice for enterprise data engineering. Made popular by Databricks and Azure Synapse which provide Spark as cloud PaaS. Spark is a distributed compute platform which can scale to handle true "big data" workloads.

3. **Polars** — a Rust-based engine with a Python API providing a powerful query engine with a DataFrame-based API. Polars leverages lazy evaluation and query optimisation to achieve high performance on single-node environments.

4. **DuckDB** — a C++ engine with a Python API providing an in-process analytical database. DuckDB is optimised for OLAP workloads and can efficiently process data directly from various file formats.

## Fabric Capacity

A Fabric capacity is a dedicated pool of compute resources that you purchase from Azure. Think of it as reserving a fixed amount of computational horsepower that's available to you continuously.

When you purchase a Fabric capacity (e.g., F8, F64), you're setting the number of capacity units (CUs) you can consume.

For example, an F8 capacity provides 8 CUs that are continuously available regardless of whether they're actively being used.

The capacity is shared across all workspaces assigned to it, and all Fabric workloads (notebooks, pipelines, warehouses, Power BI, etc.) draw from this same pool.

## Fabric Capacity Units (CUs)

CUs are Fabric's abstraction layer for billing compute across heterogeneous workloads. Different engines have different conversion rates.

This means your F64 capacity represents different amounts of "power" depending on which engine is consuming it. The CU is the common denominator for billing purposes.

The rate at which a Fabric workload will draw from the available capacity is dependent on how that workload is sized. Consumption is measured in **CU Seconds**: the number of CUs consumed multiplied by the duration in seconds.

The fundamental formula for both Spark Notebooks and Python Notebooks in Fabric is: 0.5 CU per second per vCore.

For a Python Notebook the CU per second is a simple calculation.

For a Spark Notebook the calculation is a little more complex because we need to calculate the number vCores across the single driver and potentially multiple executor nodes.

For example, a Spark Notebook with 1 driver with 8 vCores and 4 exectors also with 8 vCores: the total number of vCores = (1 + 4) * 8 vCores = 40 vCores.  Therefore the CUs Per Second for this size of Spark Notebook is 20.


## Configurations

It is difficult to achieve parity across the Spark and Python notebook platforms.  We opted for the following configurations which enabled us to achieve a range of different capacity unit (CU) per second configurations where in some cases we are able to compare like with like.

| CUs Per Second | Python Notebook Configuration | Spark Pool Configuration |
| --- | ---                 | --- |
| 1  | **2 vCores, 16G RAM** [default]   |  |
| 2   | 4 vCores, 32G RAM   |  |
| 4   | 8 vCores, 64G RAM  | 1 Executor 4/4 vCores 28G/28G RAM |
| 6   |  | 2 Executors 4/4 vCores 28G/28G RAM |
| 8   | 16 vCores, 128G RAM | **1 Executor 8/8 vCores 56G/56G RAM** [default] |
| 10  |  | 4 Executors 4/4 vCores 28G/28G RAM |
| 12  |  | 2 Executors 8/8 vCores 56G/56G RAM |
| 16  | 32 vCores, 256G RAM |  |
| 20  |  | 4 Executors 8/8 vCores 56G/56G RAM |

To keep the charts as compact as possible, we have constrained our analysis to: 1, 2, 4, 8, 12, 16 and 20 CUs Per Second sized environments.

The default configuration for a Pyspark notebook is `1 Executor 8/8 vCores 56G/56G RAM`.

The default configuration for a Python notebook is `2 vCores, 16G RAM`.

What is interesting is that the default Python Notebook (`2 vCores, 16G RAM`) is 8X cheaper to run than the default Spark Notebook (`1 Executor 8/8 vCores 56G/56G RAM`).

The smallest Spark Notebook configuration (`1 Executor 4/4 vCores 28G/28G RAM`) is equivalent to the `8 vCores, 64G RAM` which has ~2.5 X the RAM, and as we'll show significant computation horsepower if you choose to use modern "in process" query engines.

The spin up time for these configurations of platform were significantly faster (as we can see in the analysis below) which means that they should be the default choice for development purposes with the option to scale up when using at production scale.

In [15]:
spin_up_times = (
    benchmarks
    .filter(pl.col("stage_name") == "setup")
)

In [16]:
(
    spin_up_times
    .group_by(["platform", "configuration"])
    .agg(pl.col("stage_time_delta").median().alias("median_provisioning_time"))
    .sort("median_provisioning_time")
)

platform,configuration,median_provisioning_time
str,str,f64
"""Fabric PySpark Notebook""","""01 executors 08/08 cores 56g/5…",18.6
"""Fabric Python Notebook""","""02 vCores""",24.356
"""Fabric Python Notebook""","""16 vCores""",125.15
"""Fabric Python Notebook""","""08 vCores""",129.215
"""Fabric Python Notebook""","""32 vCores""",132.767
"""Fabric Python Notebook""","""04 vCores""",136.2435
"""Fabric PySpark Notebook""","""02 executors 08/08 cores 56g/5…",188.749
"""Fabric PySpark Notebook""","""04 executors 08/08 cores 56g/5…",192.767
"""Fabric PySpark Notebook""","""01 executors 04/04 cores 28g/2…",194.956


In [17]:
fig = px.box(spin_up_times, 
             x="configuration",
             y="stage_time_delta",
             title="Time to Provision Environment",
             color="platform"
             )

fig.update_layout(
    yaxis_title="Elapsed Time (seconds)"
)

fig.show(config={"toImageButtonOptions": {"format": "png", "scale": 2, "filename": "chart1", "width": 1920, "height": 1080}})

## Methodology

Multiple runs were completed for each combination of: Engine, Workload and Configuration to enable median times to be calculated.

## Analysis

## Elapsed Time Analysis

Elapsed time analysis **includes** the time to spin up the required Spark or Python environment on which the workload is running.

The "default" environments for Spark and Python therefore have an advantage in this analysis because they take ~3 and ~2 minutes less time to provision respectively.

Note - the "spin up time" for an environment does not incur a CU cost on Fabric.  So if cost rather than execution time is your primary objective, fast forward to the Execution Time analysis below!

In [18]:
elapsed_time_benchmarks = (
    benchmarks
    .sort(["run_timestamp", "order"])
    .group_by(["platform", "configuration", "workload_name", "run_timestamp", "cu_per_second"])
    .agg(pl.col("stage_time_delta").sum().alias("total_elapsed_time"))
    .sort(["platform", "configuration", "workload_name", "run_timestamp"])
)

In [19]:
summarised_overall_benchmarks = (
    elapsed_time_benchmarks
    .group_by(["platform", "configuration", "workload_name", "cu_per_second"])
    .agg(pl.col("total_elapsed_time").median().alias("median_elapsed_time"))
    .sort(["platform", "configuration", "workload_name", "cu_per_second"])
)

The table below shows the median elapsed time for the different workloads on the default environments.

Observations:
- Pyspark is the fastest (~126 seconds): on a single executor environment with 8 cores and 56GB RAM on both drive and executor nodes.
- DuckDB (~ 133 seconds) and Polars (~174 seconds) come a close second and third: on a single node with 2 cores and 16GB of RAM.
- DuckDB and Polars achieves comparable performance to Spark at an **eighth of the CU cost**.
- Pandas comes last (~275 seconds) taking roughly twice as long to complete the task.

If you take into account that the spin up time for the "default" Python environment is on average ~6 seconds longer than the "default" Spark environment, DuckDB is running in an equivalent time on a much smaller footprint, and Polars is not far behind.

In [20]:
# Filter elapsed time to "default" environments
(
    summarised_overall_benchmarks
    .filter(pl.col("configuration").is_in(['01 executors 08/08 cores 56g/56g memory', '02 vCores']))
    .sort("median_elapsed_time")
    .with_columns(
        ((pl.col("median_elapsed_time") / pl.col("median_elapsed_time").min()) * 100).round(1).alias("percentage_of_min_time")
    )
)

platform,configuration,workload_name,cu_per_second,median_elapsed_time,percentage_of_min_time
str,str,str,i64,f64,f64
"""Fabric PySpark Notebook""","""01 executors 08/08 cores 56g/5…","""pyspark_benchmark""",8,125.691,100.0
"""Fabric Python Notebook""","""02 vCores""","""duckdb_benchmark""",1,133.59,106.3
"""Fabric Python Notebook""","""02 vCores""","""polars_benchmark""",1,174.876,139.1
"""Fabric Python Notebook""","""02 vCores""","""pandas_benchmark""",1,275.515,219.2


## Execution Time Analysis

The execution time excludes the time to spin up the environment and set up the notebook (e.g. importing Python packages).

This removes the nuances of Fabric environment provisioning out of the picture, enabling a direct comparison between the different engines (Spark, DuckDB, Polars and Pandas).

The **Top 10** results are presented below. Key observations:

- DuckDB achieves the fastest execution time on a single node with 8 cores and 64GB of RAM.
- DuckDB and Polars occupy the top 8 spots before Spark appears.
- The fastest Spark execution time is more than twice that of DuckDB, which achieves its best result on infrastructure with half the resources.

In [21]:
overall_benchmarks_excluding_start_and_setup = (
    benchmarks
    .filter(pl.col("phase") != "start_and_setup")
    .sort(["run_timestamp", "order"])
    .group_by(["platform", "configuration", "workload_name", "run_timestamp", "cu_per_second"])
    .agg(pl.col("stage_time_delta").sum().alias("total_execution_time"))
    .sort(["platform", "configuration", "workload_name", "run_timestamp"])
)

In [22]:
sumamrised_overall_benchmarks_excluding_start_and_setup = (
    overall_benchmarks_excluding_start_and_setup
    .group_by(["platform", "configuration", "workload_name", "cu_per_second"])
    .agg(pl.col("total_execution_time").median().alias("median_execution_time"))
    .sort(["platform", "configuration", "workload_name", "cu_per_second"])
)

In [23]:
(
    sumamrised_overall_benchmarks_excluding_start_and_setup
    .sort("median_execution_time")
    .with_columns(
        ((pl.col("median_execution_time") / pl.col("median_execution_time").min()) * 100).round(1).alias("percentage_of_min_time")
    )
    .with_row_index("rank", offset=1)
    .head(10)
)

rank,platform,configuration,workload_name,cu_per_second,median_execution_time,percentage_of_min_time
u32,str,str,str,i64,f64,f64
1,"""Fabric Python Notebook""","""08 vCores""","""duckdb_benchmark""",4,47.037,100.0
2,"""Fabric Python Notebook""","""04 vCores""","""duckdb_benchmark""",2,66.405,141.2
3,"""Fabric Python Notebook""","""16 vCores""","""duckdb_benchmark""",8,72.794,154.8
4,"""Fabric Python Notebook""","""32 vCores""","""duckdb_benchmark""",16,80.182,170.5
5,"""Fabric Python Notebook""","""16 vCores""","""polars_benchmark""",8,86.28,183.4
6,"""Fabric Python Notebook""","""08 vCores""","""polars_benchmark""",4,88.203,187.5
7,"""Fabric Python Notebook""","""32 vCores""","""polars_benchmark""",16,90.787,193.0
8,"""Fabric Python Notebook""","""04 vCores""","""polars_benchmark""",2,102.398,217.7
9,"""Fabric PySpark Notebook""","""01 executors 08/08 cores 56g/5…","""pyspark_benchmark""",8,107.091,227.7


The line chart below shows median execution times for each engine running on different configurations, with CUs per second as a common measure of environment size and cost.

Key observations:

- The execution time of all engines initially decreases as more cores/memory is made available, but there appears to be a "sweet spot" beyond which execution times plateau or even increase slightly.
- This highlights a key point: throwing more infrastructure resources at a process will not necessarily make it faster—it could actually make it slower!
- At comparable CU levels (e.g., 4 CUs), DuckDB and Polars on Python notebooks significantly outperform PySpark on Spark notebooks.



In [24]:
fig = px.line(sumamrised_overall_benchmarks_excluding_start_and_setup, 
             x="cu_per_second",
             y="median_execution_time",
             color="workload_name",
             markers=True,
             title="Total Execution Time (excluding Setup)",
             category_orders={
                 "cu_per_second": [1, 2, 4, 8, 12, 16, 20],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"]
                 }
             )

# Map workload colors (from my_palette) with 50% transparency
workload_colors = {
    "pandas_benchmark": "rgba(132, 189, 0, 0.5)",     # #84BD00
    "pyspark_benchmark": "rgba(232, 119, 34, 0.5)",   # #E87722
    "polars_benchmark": "rgba(65, 182, 230, 0.5)",    # #41B6E6
    "duckdb_benchmark": "rgba(49, 17, 74, 0.5)",      # #31114A
}

for row in sumamrised_overall_benchmarks_excluding_start_and_setup.iter_rows(named=True):
    fig.add_annotation(
        x=row["cu_per_second"],
        y=row["median_execution_time"],
        text=row["configuration"],
        textangle=-90,
        showarrow=False,
        yanchor="bottom",
        yshift=5,
        font=dict(size=8),
        bgcolor=workload_colors.get(row["workload_name"], "rgba(255,255,255,0.5)")
    )

fig.update_layout(
    yaxis_title="Total Execution Time (seconds)"
)


fig.show(config={"toImageButtonOptions": {"format": "png", "scale": 2, "filename": "chart4", "width": 1920, "height": 1080}})

## CU Cost Analysis

Now we shift the focus from execution time to cost as the primary concern.

The following table lists the 10 cheapest engine / configuration combinations.

The cheapest Spark run is more than 5 times the cost of the cheapest DuckDB run and circa 4 times the cost of the cheapest Polars run.

In [25]:
cu_cost = (
    overall_benchmarks_excluding_start_and_setup
    .group_by(["platform", "configuration", "workload_name", "cu_per_second"])
    .agg(pl.col("total_execution_time").median().alias("median_execution_time"))
    .with_columns(
        (pl.col("median_execution_time") * pl.col("cu_per_second")).alias("total_cu_cost") )
    .sort(["platform", "configuration", "workload_name", "cu_per_second"])
)

In [26]:
(
    cu_cost
    .sort("total_cu_cost")
    .with_columns(
        ((pl.col("total_cu_cost") / pl.col("total_cu_cost").min()) * 100).round(1).alias("percentage_of_min_cost")
    )
    .with_row_index("rank", offset=1)
    .head(10)
    .select(["rank", "workload_name", "configuration", "cu_per_second", "median_execution_time", "total_cu_cost", "percentage_of_min_cost"])
)

rank,workload_name,configuration,cu_per_second,median_execution_time,total_cu_cost,percentage_of_min_cost
u32,str,str,i64,f64,f64,f64
1,"""duckdb_benchmark""","""02 vCores""",1,107.871,107.871,100.0
2,"""duckdb_benchmark""","""04 vCores""",2,66.405,132.81,123.1
3,"""polars_benchmark""","""02 vCores""",1,156.885,156.885,145.4
4,"""duckdb_benchmark""","""08 vCores""",4,47.037,188.148,174.4
5,"""polars_benchmark""","""04 vCores""",2,102.398,204.796,189.9
6,"""pandas_benchmark""","""02 vCores""",1,250.84,250.84,232.5
7,"""polars_benchmark""","""08 vCores""",4,88.203,352.812,327.1
8,"""pandas_benchmark""","""04 vCores""",2,224.122,448.244,415.5
9,"""duckdb_benchmark""","""16 vCores""",8,72.794,582.352,539.9


The following chart shows relative costs across all combinations of engine and configuration that we tested.

In [27]:
fig = px.bar(cu_cost, 
             x="workload_name",
             y="total_cu_cost",
             color="workload_name",
             title="Execution Cost",
             facet_col="cu_per_second",
             category_orders={
                 "cu_per_second": [1, 2, 4, 8, 12, 16, 20],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"]
                 }
             )

fig.update_layout(
    yaxis_title="Total CU Cost"
)

fig.update_xaxes(visible=False)

fig.show(config={"toImageButtonOptions": {"format": "png", "scale": 2, "filename": "chart5", "width": 1920, "height": 1080}})

## Analysing Execution Time versus Cost

The following scatter chart shows a two-dimensional analysis across all engine/configuration combinations tested.

The horizontal x-axis shows execution time in seconds, while the vertical y-axis shows cost on a logarithmic scale.

The sweet spot is in the **bottom-left quadrant** of the chart where you get the best combination of low cost and fast execution time. We can see that this space is dominated by DuckDB and Polars.

In [28]:
cu_cost.columns

['platform',
 'configuration',
 'workload_name',
 'cu_per_second',
 'median_execution_time',
 'total_cu_cost']

In [29]:
overall_benchmarks_median = (
    elapsed_time_benchmarks
    .group_by(["platform", "configuration", "workload_name", "cu_per_second"])
    .agg(pl.col("total_elapsed_time").median().alias("median_total_elapsed_time"))
    .sort(["platform", "configuration", "workload_name", "cu_per_second"])
)

In [30]:
overall_benchmarks_median.columns

['platform',
 'configuration',
 'workload_name',
 'cu_per_second',
 'median_total_elapsed_time']

In [31]:
elapsed_time_cost = (
    overall_benchmarks_median
    .join(cu_cost, on=["platform", "configuration", "workload_name", "cu_per_second"])
)

In [32]:
overall_benchmarks_excluding_start_and_setup_median = (
    overall_benchmarks_excluding_start_and_setup
    .group_by(["platform", "configuration", "workload_name", "cu_per_second"])
    .agg(pl.col("total_execution_time").median().alias("median_total_execution_time"))
    .sort(["platform", "configuration", "workload_name", "cu_per_second"])
)

In [33]:
execution_time_cost = (
    overall_benchmarks_excluding_start_and_setup_median
    .join(cu_cost, on=["platform", "configuration", "workload_name", "cu_per_second"])
)

In [35]:
execution_time_cost

platform,configuration,workload_name,cu_per_second,median_total_execution_time,median_execution_time,total_cu_cost
str,str,str,i64,f64,f64,f64
"""Fabric PySpark Notebook""","""01 executors 04/04 cores 28g/2…","""pyspark_benchmark""",4,149.6195,149.6195,598.478
"""Fabric PySpark Notebook""","""01 executors 08/08 cores 56g/5…","""pyspark_benchmark""",8,107.091,107.091,856.728
"""Fabric PySpark Notebook""","""02 executors 08/08 cores 56g/5…","""pyspark_benchmark""",12,132.174,132.174,1586.088
"""Fabric PySpark Notebook""","""04 executors 08/08 cores 56g/5…","""pyspark_benchmark""",20,133.041,133.041,2660.82
"""Fabric Python Notebook""","""02 vCores""","""duckdb_benchmark""",1,107.871,107.871,107.871
"""Fabric Python Notebook""","""02 vCores""","""pandas_benchmark""",1,250.84,250.84,250.84
"""Fabric Python Notebook""","""02 vCores""","""polars_benchmark""",1,156.885,156.885,156.885
"""Fabric Python Notebook""","""04 vCores""","""duckdb_benchmark""",2,66.405,66.405,132.81
"""Fabric Python Notebook""","""04 vCores""","""pandas_benchmark""",2,224.122,224.122,448.244


In [58]:
fig = px.scatter(
    execution_time_cost,
    x="median_total_execution_time",
    y="total_cu_cost",
    log_y=True,
    range_x=[0, 350],
    range_y=[50, 5000],
    color="workload_name",
    symbol="workload_name",
    text="configuration",  # Add text labels directly
    title="Execution Cost vs Execution Time",
    labels={
        "median_total_execution_time": "Median Time (s)",
        "total_cu_cost": "Total CU Cost",
    },
    category_orders={
        "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"]
    },
    hover_data={
        "cu_per_second": True,
        "total_cu_cost": ":.1f",
        "workload_name": True,
        "configuration": True
    }
)

fig.update_traces(
    marker=dict(size=12),
    textposition="top center",
    textfont=dict(size=8)
)

fig.add_shape(
    type="rect",
    xref="x",
    yref="y",
    x0=0, y0=100,
    x1=150, y1=300,
    line=dict(color="red", width=2),
    fillcolor="rgba(255, 0, 0, 0.1)"
)

fig.show()
# fig.show(config={"toImageButtonOptions": {"format": "png", "scale": 2, "filename": "chart6b", "width": 1920, "height": 1080}})

## Stage Analysis

The stage analysis looks deeper into the processing stages.

Here we are specifically interested what separates DuckDB from Polars.  Both are modern "in process" query engines developed for exactly this use case.  Where is DuckDB managing to gain an edge on Polars?

It appears it is consistently in the stages of the process which involve reading from the Fabric lakehouse.

We know that DuckDB uses it's own native engine to read Delta format from Fabric lakehouse whereas Polars uses the `deltars` package for this purpose.  Is there an opportunity for Polars to adopt the same approach as DuckDB and develop its own reader?

In [ ]:
stage_cumulative_time = (
    benchmarks
    .filter(pl.col("phase") != "start_and_setup")
    .sort(["run_timestamp", "order"])
    .group_by(["platform", "configuration", "workload_name", "cu_per_second", "stage_name", "order"])
    .agg(pl.col("cumulative_time").median().alias("median_cumulative_time"))
    .sort(["platform", "configuration", "workload_name", "cu_per_second", "order"])
)  

In [ ]:
fig = px.line(
    stage_cumulative_time,
    x="stage_name", 
    y="median_cumulative_time",
    color="workload_name",  # Different line color per workload_name
    facet_col="cu_per_second",  # Different dash pattern per t_shirt_size
    markers=True,
    title="Median Cumulative Time by Stage and Configuration",
    category_orders={
                 "cu_per_second": [1, 2, 4, 8, 12, 16, 20],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"],
                 "stage_name": ["ingest", "transform", "write_prices", "write_dates", "write_locations", "read_prices", "read_dates", "join_and_summarise"]
                 }
)
fig.show(config={"toImageButtonOptions": {"format": "png", "scale": 2, "filename": "chart7", "width": 1920, "height": 1080}})

## Conclusions

This benchmarking study reveals several important findings for teams choosing data processing engines on Microsoft Fabric:

1. **Modern in-process engines outperform Spark for medium-scale workloads** — DuckDB and Polars consistently delivered faster execution times than PySpark across all comparable configurations, often by a factor of 2x or more.- Profile your specific workload to find the optimal resource configuration rather than defaulting to larger instances.  Reserve Spark for truly distributed workloads that exceed single-node memory capacity.

2. **Cost efficiency favours Python notebooks with modern engines** — the cheapest configurations were DuckDB and Polars running on Python notebooks with minimal vCores. The cheapest Spark configuration costs 4-5x more than the cheapest DuckDB/Polars runs.- Start with smaller configurations during development—they're faster to provision and cheaper to run. For datasets that fit in memory (up to tens of GB), consider DuckDB or Polars on Python notebooks as your default choice.

3. **More resources don't always mean faster execution** — there's a "sweet spot" for resource allocation beyond which performance plateaus or degrades. This challenges the common assumption that scaling up infrastructure will proportionally improve performance.

4. **Default configurations are not equal** — the default Python notebook (2 vCores) is 8x cheaper to run than the default Spark notebook, yet achieves comparable or better performance with DuckDB/Polars.

5. **DuckDB's native Delta reader provides an edge** — Stage-level analysis suggests DuckDB's advantage over Polars comes primarily from its native Delta file reading capabilities.